In [5]:
import pandas as pd
import numpy as np

In [6]:
COLS_CLIMA = ["temp", "temp_max", "temp_min", "umidade"]

def get_climatological_mean(df, cols, target_date, window=7, offsets=(-1,1,-2,2)):
    """Média dos mesmos dias em anos adjacentes (janela ±window dias)."""
    frames = []
    for offset in offsets:
        yr = target_date.year + offset
        try:
            center = target_date.replace(year=yr)
        except ValueError:                          # 29/fev em ano não-bissexto
            center = target_date.replace(year=yr, day=28)
        sub = df.loc[center - pd.Timedelta(days=window):
                     center + pd.Timedelta(days=window), cols].dropna(how="all")
        if len(sub) >= 3:
            frames.append(sub)
    return pd.concat(frames).mean() if frames else None


def impute_partial_maxmin(df):
    """Lacuna E: estima temp_max/min a partir da temp presente + amplitude histórica."""
    hist = df[(df.index.month == 11) & (df.index.year < 2025)]
    delta_max = (hist["temp_max"] - hist["temp"]).mean()
    delta_min = (hist["temp"] - hist["temp_min"]).mean()

    mask = (
        (df.index >= "2025-11-02") &
        (df.index <= "2025-11-16") &
        df["temp_max"].isna() &
        df["temp"].notna()
    )
    df.loc[mask, "temp_max"] = (df.loc[mask, "temp"] + delta_max).round(2)
    df.loc[mask, "temp_min"] = (df.loc[mask, "temp"] - delta_min).round(2)
    print(f"  Lacuna E: {mask.sum()} dias preenchidos via amplitude histórica "
          f"(Δmax=+{delta_max:.2f}°C, Δmin=-{delta_min:.2f}°C)")
    return df


def impute_all(df: pd.DataFrame, verbose=True) -> pd.DataFrame:
    df = df.copy()

    # Passo 1: lacuna parcial de temp_max/min em Nov/2025
    df = impute_partial_maxmin(df)

    # Passo 2: média climatológica para todas as lacunas restantes
    remaining = df[COLS_CLIMA].isna().any(axis=1)
    dates_to_fill = df.index[remaining]

    if verbose:
        print(f"\nImputação climatológica: {len(dates_to_fill)} dias restantes\n")

    filled, failed = 0, []
    for date in dates_to_fill:
        # Para 2025 usa só anos anteriores (sem futuro disponível)
        offsets = (-1, -2, -3) if date.year == 2025 else (-1, 1, -2, 2)
        mean = get_climatological_mean(df, COLS_CLIMA, date, window=7, offsets=offsets)

        if mean is not None:
            for col in COLS_CLIMA:
                if pd.isna(df.loc[date, col]) and not pd.isna(mean[col]):
                    df.loc[date, col] = round(mean[col], 3)
            filled += 1
        else:
            failed.append(date)

    if verbose:
        print(f"  Preenchidos: {filled} dias")
        if failed:
            print(f"  Sem referência: {[d.date() for d in failed]}")

    # Passo 3: verifica se ainda sobrou algum NaN
    still_nan = df[COLS_CLIMA].isna().sum()
    if still_nan.sum() > 0:
        print(f"\n[AVISO] NaN remanescentes:\n{still_nan[still_nan > 0]}")
    else:
        print("\n✓ Dataset completamente imputado, zero NaN.")

    return df


# ── Uso ────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_csv(
        "../data/weather.csv",
        parse_dates=["datetime"],
        index_col="datetime"
    )

    print(f"NaN antes:\n{df[COLS_CLIMA].isna().sum()}\n")
    df_imp = impute_all(df, verbose=True)
    print(f"\nNaN depois:\n{df_imp[COLS_CLIMA].isna().sum()}")

    # Salva
    df_imp.to_csv("../data/climate_imputed.csv")
    print("\nSalvo em data/climate_imputed.csv")


NaN antes:
temp        180
temp_max    192
temp_min    192
umidade     180
dtype: int64

  Lacuna E: 12 dias preenchidos via amplitude histórica (Δmax=+6.24°C, Δmin=-4.71°C)

Imputação climatológica: 180 dias restantes

  Preenchidos: 180 dias

✓ Dataset completamente imputado, zero NaN.

NaN depois:
temp        0
temp_max    0
temp_min    0
umidade     0
dtype: int64

Salvo em data/climate_imputed.csv
